# 21.20 / MP-08. Реконструкция 4D-КТ: проверка входов

**Статус: технический preflight реальных файлов и ограниченный синтетический API smoke; пациентская реконструкция не выполнялась.** Проверяемый вопрос: существует ли реальная многофазная КТ с согласованной геометрией и независимая опорная КТ большего разрешения, а также прошла ли регистрация MP-03 QC. Положительный результат будущего опыта означал бы уменьшение заранее выбранных ошибок фазовых геометрических измерений относительно исходных фаз при независимом контроле. Отрицательный результат — отсутствие выигрыша, артефакты или неприемлемое искажение. Пороги нужно определить до расчёта.

Первичный метод: [MONAI Physio Tutorial 3](https://project-monai.github.io/monai-physio/tutorials.html#tutorial-3-reconstruct-high-resolution-4d-ct) и [WorkflowReconstructHighres4DCT](https://project-monai.github.io/monai-physio/api/workflows.html#high-resolution-4d-ct-reconstruction). Workflow регистрирует фазы к опорному изображению и реконструирует их на его сетке. Название high-resolution само по себе не подтверждает добавление измеренной пространственной информации.

## Основной MP-08 preflight на Нике

[Read-only повтор](mp08_09_nix_preflight_results.json) выполнен на 12 реальных фазах Ника. Все заголовки имеют общую сетку `512×512×256`, шаг по affine `0,765625×0,765625×0,625` мм, согласованные sform/qform и манифест конверсии. Единицы заголовка NIfTI — `unknown`; миллиметры следуют из DICOM-конверсии. Отдельная опора более высокого разрешения не предъявлена. [Регистрация 00→01/07](mp03_nix_fullres_report.md) технически рассчитана, но не принята по независимой анатомии и FOV. Поэтому реконструкция пациентской 4D-КТ не выполнялась. Старый preflight Георга ниже оставлен как исторический материал.


## Прочитанные входы и результат

**Факт о файлах, 28.09.2026.** Из внешнего производного хранилища на локальной копии D: прочитаны конверсионный манифест и только заголовки 15 NIfTI фаз испытуемого с псевдонимом Georg. Пиксели, исходные DICOM и другие зеркала не читались и не копировались. Фазы имеют сетку 512 × 512 × 256 и одинаковый affine; шаги из affine — 0.458984 × 0.458984 × 0.625 мм по конверсионному манифесту. Статус манифеста: `converted_pending_image_qc`. Сведения о физической единице в NIfTI-заголовке имеют значение `unknown`; миллиметры пока опираются на происхождение конверсии, а не на единицу NIfTI. Проверка sform/qform и совпадения манифеста с заголовками прошла. Программа и обезличенный агрегированный вывод: `mp08_09_preflight.py`, `mp08_09_preflight_results.json`.

**Временное ограничение.** Тег фаз — `NominalPercentageOfCardiacPhase`, значения 23–164; это не миллисекунды и не автоматически один равномерный механический цикл. Для Georg в проекте отмечены два различных R–R интервала 491 и 1159 мс. Привязка и выбор анализируемого интервала требуют отдельного принятия.

**Нерешённый вопрос.** Отдельный индивидуальный том с более мелкой сеткой для этой постановки не предоставлен. Использование одной из этих фаз как опоры оставляет разрешение сетки тем же. MP-03 ещё не передал принятое поле и QC складок, обратимости, границ и анатомического совмещения. Поэтому пациентская реконструкция не запускалась.

## Синтетический API smoke (28.09.2026)

**Модельный результат.** В установленном MONAI Physio 2026.09.1 `mp08_09_synthetic_smoke.py` создал два аналитических 3D фантома 32³ с шагом 1 мм и отдельную опору 48³ с шагом 0.75 мм. Известный сдвиг второго фантома — (2, −1, 0) мм. Вызван официальный `WorkflowReconstructHighres4DCT` с `RegisterImagesGreedy` в rigid-режиме и коротким расписанием [8, 4, 2]. API вернул два реконструированных объёма 43³ с шагом 0.75 мм и трансформы. Размер 43³ отражает выбранный workflow/FOV, а не совпадает автоматически с полной сеткой опоры 48³.

**Ограничение.** В нескольких запусках короткого smoke норма ошибки восстановления заданного сдвига в центре менялась примерно от 1.05 до 1.20 мм (первый запуск — 1.2004 мм); конкретное значение и вектор последнего запуска записаны в `mp08_09_synthetic_results.json`. При столь короткой оптимизации точного восстановления заданного движения **не показано**. Число не переносится на пациента и не является оценкой точности MONAI Physio в клинике. Фантом создан в памяти, медицинские данные не читались.

Для отдельной проверки точности алгоритма потребуются заранее заданные допуски, несколько сдвигов/деформаций и настройки оптимизации; этот запуск подтверждает только работоспособность данного маршрута API и фиксирует его ошибку.

## Следующий вычислительный опыт после QC

Зафиксировать опорную фазу и её выбор, дыхательное состояние, реальную временную ось, версию MONAI Physio и регистратора, исходные контрольные суммы во внешнем локальном журнале. На малой заранее выбранной паре фаз сравнить исходное изображение, простое пересэмплирование без деформации и регистрационную реконструкцию на одной и той же сетке. Отдельно оценить складки и направление поля, обратную согласованность, сохранность HU и границ, объёмы и расстояния поверхностей; затем выполнить слепой визуальный QC. Измерение на уже использованных при регистрации изображениях не является независимой валидацией. До выяснения ошибок сегментации, регистрации и временного сопоставления не переносить результат в УО/ФВ или FEM/ТТРКГ.

**Источники погрешности:** неодинаковая фаза дыхания/контраст, движение и артефакты исходной КТ, интерполяция, регистрация, выбор референса, сегментация и нарушение соответствия двух сердечных интервалов. Их распределение и независимость не заданы. Кодовый самоконтроль заголовков не оценивает эти компоненты.